In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from src.arhitecture import TverskyProjection

In [2]:
torch.manual_seed(42)

X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y = torch.tensor([0, 1, 1, 0])

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

model = TverskyProjection(
    in_features=2,
    out_features=2,
    num_features=2,
    model_type='contrast',
    intersection_reduction='product', # product обычно дает более стабильные градиенты
    difference_type='subtractmatch'
)

In [3]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

epochs = 1000
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        
        outputs = model(batch_X)
        
        loss = criterion(outputs, batch_y)
        loss.backward()
        
        # Шаг оптимизации
        optimizer.step()
        total_loss += loss.item()
        
    scheduler.step()
    
    if (epoch + 1) % 100 == 0:
        model.eval()
        with torch.no_grad():
            logits = model(X)
            predictions = torch.argmax(logits, dim=1)
            accuracy = (predictions == y).float().mean().item() * 100.0
            print(f"Epoch {epoch+1:4d}/{epochs} | Loss: {total_loss:.4f} | Acc: {accuracy:5.1f}%")
            
model.eval()
with torch.no_grad():
    final_logits = model(X)
    final_preds = torch.argmax(final_logits, dim=1)
    print("\n=== Final Results ===")
    for i in range(4):
        print(f"Input: {X[i].tolist()} -> Prediction: {final_preds[i].item()} (True: {y[i].item()})")


Epoch  100/1000 | Loss: 0.5260 | Acc:  75.0%
Epoch  200/1000 | Loss: 0.1710 | Acc: 100.0%
Epoch  300/1000 | Loss: 0.0653 | Acc: 100.0%
Epoch  400/1000 | Loss: 0.0368 | Acc: 100.0%
Epoch  500/1000 | Loss: 0.0255 | Acc: 100.0%
Epoch  600/1000 | Loss: 0.0201 | Acc: 100.0%
Epoch  700/1000 | Loss: 0.0175 | Acc: 100.0%
Epoch  800/1000 | Loss: 0.0161 | Acc: 100.0%
Epoch  900/1000 | Loss: 0.0156 | Acc: 100.0%
Epoch 1000/1000 | Loss: 0.0155 | Acc: 100.0%

=== Final Results ===
Input: [0.0, 0.0] -> Prediction: 0 (True: 0)
Input: [0.0, 1.0] -> Prediction: 1 (True: 1)
Input: [1.0, 0.0] -> Prediction: 1 (True: 1)
Input: [1.0, 1.0] -> Prediction: 0 (True: 0)
